# Bars inside bars

A bar that starts while another one runs becomes its child: it is drawn under its parent,
indented, in the same output. Nothing needs to be passed along.

In [ ]:
import time
from concurrent.futures import ThreadPoolExecutor

from anyprogressbar import progress

In [ ]:
for epoch in progress(range(3), "Epochs"):
    for batch in progress(range(20), f"Batches of epoch {epoch}", unit="batches"):
        time.sleep(0.04)

A child that is done leaves the picture, so a long job does not grow a long list. To keep
one, pass `leave=True`:

In [ ]:
for name in progress(["train", "validate", "test"], "Datasets"):
    for row in progress(range(15), name, unit="rows", leave=True):
        time.sleep(0.04)

## Functions that report their own progress

This is what makes children useful: a function can show a bar without knowing who calls
it. Called on its own it gets a bar of its own; called inside a loop with a bar, it appears
under that one.

In [ ]:
def convert(name, pages=12):
    for page in progress(range(pages), f"Converting {name}", unit="pages"):
        time.sleep(0.04)

convert("report.pdf")

In [ ]:
for name in progress(["a.pdf", "b.pdf", "c.pdf"], "Files", unit="files"):
    convert(name)

## Naming the parent

Work in another thread does not know which bar it belongs to. Say so with `parent=`.

In [ ]:
def fetch(site, parent):
    with progress(total=10, desc=site, unit="pages", parent=parent) as bar:
        for _ in range(10):
            time.sleep(0.06)
            bar.update()
    parent.update()

sites = ["alpha", "beta", "gamma"]
with progress(total=len(sites), desc="Sites") as overall, ThreadPoolExecutor(3) as pool:
    for site in sites:
        pool.submit(fetch, site, overall)